In [57]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import requests
import folium
from numpy import isnan

# Vienna Station Map

This small project is visualising a dataset from WienMobil (https://api-portal.wienerstadtwerke.at/portal/apis/2f86db5c-956e-471c-a741-041cfb0ef438/apiinfo). The bikesharing stations of the first nine districts of Vienna are being filtered and visualised along with their capacity on a python-generated map.

In [7]:
# Define the json file paths
info_json = "station_information.json"
status_json = "station_status.json"

info_df = pd.read_json(info_json)
status_df = pd.read_json(status_json)

In [58]:
# Function for determining marker color
def get_color(capacity):
    if isnan(capacity):
        return "gray"
    elif capacity < 15:
        return "green"
    elif capacity < 25:
        return "orange"
    else:
        return "red"

In [8]:
# Load Vienna district GeoJSON
url = "https://data.wien.gv.at/daten/geo?service=WFS&request=GetFeature&version=1.1.0&typeName=ogdwien:BEZIRKSGRENZEOGD&srsName=EPSG:4326&outputFormat=json"
geojson = requests.get(url).json()

districts = gpd.GeoDataFrame.from_features(geojson["features"])
districts = districts.set_crs("EPSG:4326")

In [14]:
# Filter districts 1–9
central_districts = districts[districts["BEZNR"].astype(int).between(1, 9)]

In [15]:
# Convert your stations to GeoDataFrame
points = gpd.GeoDataFrame(
    info_df.data.stations,
    geometry=[Point(d["lon"], d["lat"]) for d in info_df.data.stations],
    crs="EPSG:4326"
)

In [34]:
# Spatial join (point-in-polygon)
filtered_points = gpd.sjoin(points, central_districts, predicate="within")

print(filtered_points[["name", "lat", "lon"]])

# Plot with Folium
m = folium.Map(location=[48.2082, 16.3738], zoom_start=14)

                                   name        lat        lon
0                     Julius-Raab-Platz  48.211544  16.382374
1                           Hoher Markt  48.210666  16.372983
2                   Oper / Karlsplatz U  48.202683  16.369702
3                 Ring / Volkstheater U  48.206516  16.360400
4                       Stephansplatz U  48.207836  16.372111
..                                  ...        ...        ...
247                          Hafen Wien  48.183217  16.458423
248                   Hundertwasserhaus  48.207691  16.394724
257      Rebhanngasse / Nordbahnstrasse  48.227231  16.388737
258                     Nordbahnstrasse  48.224594  16.389338
259  Ziegelofengasse / Margaretenstraße  48.192035  16.359592

[93 rows x 3 columns]


In [63]:
# Add filtered stations
for _, row in filtered_points.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=12,
        color=get_color(row["capacity"]),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['name']}\n(Cap: {row['capacity']})"
    ).add_to(m)

# HTML tooltip
legend_html = """
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 180px; height: 120px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
    ">
    <b>Station Capacity</b><br>
    <i style="background:green;width:10px;height:10px;display:inline-block;"></i> Low (&lt;15)<br>
    <i style="background:orange;width:10px;height:10px;display:inline-block;"></i> Medium (15–24)<br>
    <i style="background:red;width:10px;height:10px;display:inline-block;"></i> High (25+)<br>
    <i style="background:gray;width:10px;height:10px;display:inline-block;"></i> Unknown (NAN)
</div>
"""

# Add tooltip to map
m.get_root().html.add_child(folium.Element(legend_html))

# Save map to HTML
m.save("vienna_districts_filtered.html")

print("Map saved as vienna_districts_filtered.html")

Map saved as vienna_districts_filtered.html
